In [11]:
import os, re, numpy as np
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.preprocessing import MinMaxScaler

import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pinecone import Pinecone, ServerlessSpec

import google.generativeai as genai

# ==============================
# 🔑 LOAD ENV
# ==============================
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

# ==============================
# 🤖 MODEL
# ==============================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
gemini = genai.GenerativeModel("gemini-2.5-flash-lite")

# ==============================
# 📂 LOAD PDF (ID ONLY)
# ==============================
def load_pdfs(folder):
    docs = []
    for f in os.listdir(folder):
            path = os.path.join(folder, f)
            text = pymupdf4llm.to_markdown(path)

            docs.append({
                "text": text,
                "source": f
            })
    return docs


# ==============================
# ✂️ CHUNKING
# ==============================
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

def is_noise(text):
    return "picture" in text.lower() or len(text.strip()) < 30

def chunk_docs(docs):
    chunks = []
    for d in docs:
        if not d["text"].strip():
            continue

        for c in splitter.split_text(d["text"]):
            if c.strip() and not is_noise(c):
                chunks.append({
                    "text": c,
                    "source": d["source"]
                })
    return chunks

# ==============================
# 🧠 EMBEDDING
# ==============================
def embed(texts):
    return embed_model.encode(texts, normalize_embeddings=True)

# ==============================
# 🗄️ PINECONE SETUP
# ==============================
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "mitsubishi-hybrid"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

# ==============================
# 📥 UPSERT DATA
# ==============================
def upsert(chunks, batch_size=100):
    texts = [c["text"] for c in chunks]
    vecs = embed(texts)

    data = []
    for i, c in enumerate(chunks):
        data.append({
            "id": f"id-{i}",
            "values": vecs[i].tolist(),
            "metadata": {
                "text": c["text"],
                "source": c["source"]
            }
        })

    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        index.upsert(vectors=batch)

# ==============================
# 🔍 BM25
# ==============================
def build_bm25(chunks):
    tokenized = [c["text"].lower().split() for c in chunks]
    return BM25Okapi(tokenized)

# ==============================
# 🔥 HYBRID SEARCH
# ==============================
def hybrid_search(query, chunks, bm25, top_k=5):
    q_emb = embed([query])[0]

    dense = index.query(
        vector=q_emb.tolist(),
        top_k=top_k,
        include_metadata=True
    )

    tokenized_q = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_q)

    scaler = MinMaxScaler()
    bm25_norm = scaler.fit_transform(np.array(bm25_scores).reshape(-1,1)).flatten()

    results = []

    for m in dense["matches"]:
        idx = int(m["id"].split("-")[1])

        dense_score = m["score"]
        sparse_score = bm25_norm[idx] if idx < len(bm25_norm) else 0

        score = 0.7 * dense_score + 0.3 * sparse_score

        results.append({
            "text": m["metadata"]["text"],
            "source": m["metadata"]["source"],
            "score": score
        })

    return sorted(results, key=lambda x: x["score"], reverse=True)

# ==============================
# 🧠 GEMINI ANSWER (RAG)
# ==============================
def ask_gemini(query, results):

    context = "\n\n".join([r["text"] for r in results[:3]])

    prompt = f"""
Anda adalah AI otomotif Mitsubishi.

Gunakan konteks katalog berikut untuk menjawab:

{context}

Pertanyaan:
{query}

Jawaban harus:
- Ringkas
- Informatif
- Berbasis katalog
"""

    response = gemini.generate_content(prompt)
    return response.text

# ==============================
# 🚀 PIPELINE
# ==============================
folder = "Data/mitsubishi_brchs/"  # path folder PDF

docs = load_pdfs(folder)
chunks = chunk_docs(docs)

print("Jumlah docs:", len(docs))
print("Jumlah chunks:", len(chunks))

if len(chunks) > 0:
    print("Contoh chunk:", chunks[0]["text"][:200])

bm25 = build_bm25(chunks)

upsert(chunks)

# ==============================
# 🎯 TEST QUERIES
# ==============================
queries = [
    "Detail spesifikasi Mitsubishi Destinator",
    "Mobil Mitsubishi untuk travel kapasitas besar",
    "Mobil nyaman perjalanan jauh Mitsubishi",
    "Mobil Mitsubishi irit bahan bakar",
    "Mobil Mitsubishi hybrid atau electric"
]

for q in queries:
    print("\n======================")
    print("QUERY:", q)

    search_results = hybrid_search(q, chunks, bm25)
    answer = ask_gemini(q, search_results)

    print("\nANSWER:\n", answer)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6816.24it/s]


Jumlah docs: 27
Jumlah chunks: 36
Contoh chunk: DESTINATOR SPECIFICATION ULTIMATE CVT EXCEED CVT GLS CVT<br>DIMENSION & WEIGHT<br>Overall Length mm 4680<br>Overall Width mm 1840<br>Overall Height mm 1780<br>Ground Clearance mm 244 **<br>Seating Cap

QUERY: Detail spesifikasi Mitsubishi Destinator


InvalidArgument: 400 API key expired. Please renew the API key.